# Preparação de ECG — C#

Pipeline em C#: WFDB formato 16 de um canal a 250 Hz → reamostragem para 200 Hz → high-pass Butterworth (5.ª ordem, 0,5 Hz) e notch (60 Hz, Q=35) → picos R → correspondência 1:1 com N/S/V/Q (100 ms) → janelas de 1 segundo → interpolação para 500 pontos e min–max → SQI fuzzy → NPZ e índices CSV.

Os dados em `data/train`, `data/dev` e `data/test` já estão preparados. Para repetir a preparação, colocar os `.hea/.dat/.atr` em `data/raw/pXX/pXXXXX/` ou definir `ICENTIA_RAW_DIR`. As divisões por paciente são lidas de `data/preparation_config.json`. A saída nova é `data/reprepared/`.

Compilar com `dotnet build src/Aih.Ecg -c Release` na raiz e usar o kernel **.NET (C#)** / Polyglot Notebooks nesta pasta. A implementação está em `src/Aih.Ecg/Preparation.cs`, `SignalProcessing.cs` e `Wfdb.cs`.


In [ ]:
#r "../src/Aih.Ecg/bin/Release/net8.0/Aih.Ecg.dll"
using System;
using System.IO;
using System.Text.Json;
using Aih.Ecg;

var root = Dataset.FindRoot();
var data = Path.Combine(root, "data");
Console.WriteLine(data);


In [ ]:
var raw = Environment.GetEnvironmentVariable("ICENTIA_RAW_DIR") ?? Path.Combine(data, "raw");
var output = Path.Combine(data, "reprepared");
Console.WriteLine($"Entrada: {raw}\nSaída: {output}");


## Verificar a implementação

Os testes incluem reamostragem, filtros, deteção de picos sintéticos, associação de anotações, normalização, SQI e leitura/escrita NPZ/WFDB.


In [ ]:
SelfTest.Run();


## Preparação completa

A execução mantém-se desligada ao abrir o notebook. Para preparar todos os registos dos 20 clientes, definir `runPreparation = true`. Um limite de `1` registo por paciente permite testar uma amostra menor; `null` processa tudo. A saída deve estar vazia.

O processamento exporta índices de cliente, manifesto, sinais, contagens, estatísticas de rejeição e configuração. Os arrays são relidos e validados após a escrita. Os parâmetros e a representação estão definidos em C#; diferenças de arredondamento podem alterar decisões próximas dos limiares.


In [ ]:
bool runPreparation = false;
int? limitRecordsPerPatient = null;
if (runPreparation)
{
    var prepared = Preparation.Run(data, raw, output, limitRecordsPerPatient);
    Console.WriteLine(JsonSerializer.Serialize(prepared, new JsonSerializerOptions { WriteIndented = true }));
}
else
{
    Console.WriteLine("Preparação completa desligada. Dados preparados disponíveis em data/.");
}


## Contagens dos dados preparados

Q significa não classificável e não é automaticamente uma arritmia. O SQI usa dois detetores (gradiente suavizado e Pan–Tompkins) e os índices q/p/k/bas em janelas de 1 segundo; esta unidade de janela e os detetores constituem uma adaptação do SQI fuzzy.


In [ ]:
Console.WriteLine(File.ReadAllText(Path.Combine(data, "split_class_counts.csv")));
